# PneumoLens — Pneumonia Dataset Audit
## Phase 2: Dataset Audit and Split Integrity

**Notebook name:** `Pneumonia`  
**Project:** PneumoLens  
**Dataset:** ZhangLab / Kermany pediatric chest X-ray collection (use the chest X-ray subset of the attached dataset).

### Objectives
1. Inspect Kaggle inputs and identify the dataset source.
2. Inventory image files without extracting a large ZIP unnecessarily.
3. Separate chest X-rays from unrelated content such as OCT images.
4. Recover labels, original split names, and optional bacterial/viral labels.
5. Inspect image dimensions, modes, and readability.
6. Infer possible patient IDs and check overlap across splits.
7. Detect exact duplicates with SHA-256.
8. Export a manifest and audit report to `/kaggle/working`.


## 1. Imports and configuration

The notebook supports an extracted directory or a ZIP archive attached to Kaggle. ZIP archives are scanned through their file index; the notebook does not extract the full archive.

In [1]:
# CELL 1 — IMPORTS, ENVIRONMENT AND CONFIGURATION
import os, re, sys, json, math, hashlib, zipfile, platform, warnings
from pathlib import Path, PurePosixPath
from contextlib import contextmanager

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/pneumolens_phase2_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Leave None for automatic discovery. Otherwise enter exact path:
# DATASET_PATH = "/kaggle/input/my-dataset/rscbjbr9sj-3.zip"
# DATASET_PATH = "/kaggle/input/my-dataset/chest_xray"
DATASET_PATH = None

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
CHEST_ONLY_FILTER = True
MAX_FILES_TO_SCAN = None       # None = scan all matching image files
CHECK_IMAGE_HEADERS = True
FULL_DECODE_CHECK = False      # slower; fully decodes every selected image
COMPUTE_SHA256 = True
MAX_HASH_FILES = 20000         # safety limit for full-file hashing
HASH_CHUNK_SIZE = 1024 * 1024
N_EXAMPLE_PATHS = 20

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)
warnings.filterwarnings("ignore", category=UserWarning)

print("PneumoLens Phase 2 — Dataset Audit")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Pandas:", pd.__version__)
print("Pillow:", Image.__version__)
print("Kaggle input exists:", INPUT_ROOT.exists())
print("Audit output:", OUTPUT_DIR)

PneumoLens Phase 2 — Dataset Audit
Python: 3.13.18
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
Pandas: 2.3.3
Pillow: 13.0.0
Kaggle input exists: True
Audit output: /kaggle/working/pneumolens_phase2_audit


## 2. Inspect Kaggle inputs and select the dataset source

If more than one possible dataset is attached, the notebook stops rather than silently choosing the wrong one. Set `DATASET_PATH` in Cell 1 and rerun this cell.

In [2]:
# CELL 2 — INPUT DISCOVERY AND SOURCE SELECTION
if not INPUT_ROOT.exists():
    raise FileNotFoundError("Run this notebook in Kaggle and attach a dataset under Input.")

input_items = sorted(INPUT_ROOT.iterdir())
print("TOP-LEVEL KAGGLE INPUTS")
for p in input_items:
    if p.is_file():
        print(f"[FILE] {p} | {p.stat().st_size / (1024**3):.3f} GB")
    else:
        print(f"[DIR ] {p}")

if not input_items:
    raise FileNotFoundError("No dataset is attached. Use Add Input in Kaggle.")

def contains_chest_folder(folder):
    accepted = {"chest_xray", "chest-xray", "chest xray", "chest_x_ray", "cxr"}
    try:
        return any(p.is_dir() and p.name.lower().strip() in accepted
                   for p in folder.rglob("*"))
    except PermissionError:
        return False

def find_zips(folder):
    try:
        return sorted(folder.rglob("*.zip"))
    except PermissionError:
        return []

if DATASET_PATH is not None:
    selected_source = Path(DATASET_PATH)
    if not selected_source.exists():
        raise FileNotFoundError(f"DATASET_PATH does not exist: {selected_source}")
elif len(input_items) == 1:
    top = input_items[0]
    if top.is_file():
        selected_source = top
    elif contains_chest_folder(top):
        selected_source = top
    else:
        zips = find_zips(top)
        selected_source = zips[0] if len(zips) == 1 else top
else:
    keywords = ("pneumonia", "chest", "xray", "x-ray", "rscbj", "kermany")
    matches = [p for p in input_items if any(k in p.name.lower() for k in keywords)]
    if len(matches) == 1:
        selected_source = matches[0]
    else:
        raise RuntimeError(
            "Dataset selection is ambiguous. Set DATASET_PATH in Cell 1.\n"
            + "\n".join(str(p) for p in input_items)
        )

print("\nSELECTED SOURCE:", selected_source)
if selected_source.is_file():
    print("Size (GB):", round(selected_source.stat().st_size / (1024**3), 3))
    if selected_source.suffix.lower() != ".zip":
        raise ValueError("Selected file is not ZIP. Choose a directory or .zip file.")
elif not selected_source.is_dir():
    raise ValueError("Selected source is neither a directory nor a ZIP.")

TOP-LEVEL KAGGLE INPUTS
[DIR ] /kaggle/input/datasets

SELECTED SOURCE: /kaggle/input/datasets


## 3. Define label, split, subtype, patient-ID, and modality parsers

The parser prefers explicit folder names. Patient IDs are inferred only from recognizable filename patterns and are marked with a confidence level. A numeric token in a name such as `NORMAL-520267-0001.jpeg` is treated as low-confidence because it may be a study/image identifier rather than a patient identifier.

In [3]:
# CELL 3 — PARSING HELPERS (CORRECTED)
def path_parts(path_text):
    text = str(path_text).replace("\\", "/")
    return [p.strip() for p in PurePosixPath(text).parts if p.strip() not in ("", ".", "/")]

def is_image_file(path_text):
    return PurePosixPath(str(path_text).replace("\\", "/")).suffix.lower() in IMAGE_EXTENSIONS

def extract_label(path_text):
    """
    Identify the nearest explicit class folder first.
    This prevents a dataset root folder named 'pneumonia' from overriding
    a deeper NORMAL class folder.
    """
    parts = path_parts(path_text)
    parent_parts = [p.lower() for p in parts[:-1]]
    stem = PurePosixPath(str(path_text).replace("\\", "/")).stem.lower()

    # The class folder nearest to the image takes priority.
    for part in reversed(parent_parts):
        if part == "normal":
            return "NORMAL"
        if part == "pneumonia":
            return "PNEUMONIA"

    # Filename fallback for datasets that encode class/subtype in filenames.
    if re.match(r"^(bacteria|bacterial|virus|viral|pneumonia)([-_]|$)", stem) or re.search(
        r"(^|[_-])(bacteria|bacterial|virus|viral)([_-]|$)", stem
    ):
        return "PNEUMONIA"
    if re.match(r"^normal([-_]|$)", stem):
        return "NORMAL"

    return None

def extract_split(path_text):
    mapping = {
        "train":"train", "training":"train",
        "val":"val", "valid":"val", "validation":"val",
        "test":"test", "testing":"test"
    }
    for part in path_parts(path_text):
        if part.lower() in mapping:
            return mapping[part.lower()]
    return None

def extract_subtype(path_text):
    filename = PurePosixPath(str(path_text).replace("\\", "/")).stem.lower()
    bacterial = bool(re.search(r"(^|[_-])(bacteria|bacterial)([_-]|$)", filename))
    viral = bool(re.search(r"(^|[_-])(virus|viral)([_-]|$)", filename))

    if bacterial and not viral:
        return "BACTERIAL"
    if viral and not bacterial:
        return "VIRAL"
    if bacterial and viral:
        return "AMBIGUOUS"
    return None

def extract_group_id_candidate(path_text):
    """
    Extract a filename-derived grouping candidate.
    These IDs may represent a patient, study, or source grouping; they are
    NOT confirmed patient identifiers and must be treated as low confidence.
    """
    filename = PurePosixPath(str(path_text).replace("\\", "/")).stem
    patterns = [
        (r"(person\d+)", "person_token", "medium"),
        (r"(patient[_-]?\d+)", "patient_token", "medium"),
        (r"(subject[_-]?\d+)", "subject_token", "medium"),
        (r"(IM-\d+)(?:-\d+)?", "im_study_prefix", "medium"),
        (
            r"(?:NORMAL|BACTERIA|BACTERIAL|VIRUS|VIRAL|PNEUMONIA)[_-](\d+)-\d+",
            "filename_numeric_group_candidate",
            "low"
        )
    ]
    for pattern, method, confidence in patterns:
        match = re.search(pattern, filename, flags=re.IGNORECASE)
        if match:
            return match.group(1).lower(), method, confidence
    return None, None, "unavailable"

def modality_hint(path_text):
    """
    Classify likely modality using path and filename.
    Explicit chest-X-ray folders take priority; clear OCT/retinal folders
    are excluded.
    """
    parts = [p.lower() for p in path_parts(path_text)]

    explicit_names = {
        "chest_xray", "chest-xray", "chest xray",
        "chest_x_ray", "chest x-ray", "cxr"
    }
    has_explicit_chest_folder = any(
        p in explicit_names
        or (p.startswith("chest") and ("xray" in p or "x-ray" in p or "x_ray" in p))
        for p in parts
    )
    if has_explicit_chest_folder:
        return "CHEST_XRAY"

    first_part = parts[0] if parts else ""
    combined_collection_title = (
        "chest" in first_part and
        ("x-ray" in first_part or "xray" in first_part or "x_ray" in first_part)
    )
    oct_parts = parts[1:] if combined_collection_title else parts
    if any(
        p in {"oct", "oct2017", "retina", "cnv", "dme", "drusen"} or p.startswith("oct")
        for p in oct_parts
    ):
        return "OCT_OR_OTHER"

    filename = PurePosixPath(str(path_text).replace("\\", "/")).name.lower()
    known_cxr_name = (
        re.search(r"person\d+", filename)
        or re.search(r"IM-\d+", filename)
        or re.search(r"(NORMAL|PNEUMONIA|BACTERIA|BACTERIAL|VIRUS|VIRAL)[_-]\d+-\d+", filename)
    )
    if known_cxr_name:
        return "CHEST_XRAY"

    if extract_label(path_text) in {"NORMAL", "PNEUMONIA"} and extract_split(path_text):
        return "CHEST_XRAY"

    return "UNCERTAIN"

def make_record(relative_path, storage_type, file_size_bytes,
                absolute_path=None, archive_member=None):
    group_id, method, confidence = extract_group_id_candidate(relative_path)
    hint = modality_hint(relative_path)
    return {
        "relative_path": str(relative_path).replace("\\", "/"),
        "filename": PurePosixPath(str(relative_path).replace("\\", "/")).name,
        "extension": PurePosixPath(str(relative_path).replace("\\", "/")).suffix.lower(),
        "storage_type": storage_type,
        "absolute_path": str(absolute_path) if absolute_path is not None else None,
        "archive_member": archive_member,
        "file_size_bytes": int(file_size_bytes),
        "label": extract_label(relative_path) or "UNKNOWN",
        "split": extract_split(relative_path) or "UNKNOWN",
        "subtype": extract_subtype(relative_path) or "UNKNOWN",
        "filename_group_id_candidate": group_id,
        "group_id_method": method,
        "group_id_confidence": confidence,
        "modality_hint": hint,
        "chest_xray_candidate": hint == "CHEST_XRAY"
    }

# Parser sanity checks using the actual path patterns observed in this dataset.
parser_examples = [
    (
        "srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1003233-0001.jpeg",
        "NORMAL"
    ),
    (
        "srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/PNEUMONIA/BACTERIA-266879-0001.jpeg",
        "PNEUMONIA"
    ),
    (
        "srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/test/PNEUMONIA/VIRUS-490019-0001.jpeg",
        "PNEUMONIA"
    )
]

for example_path, expected_label in parser_examples:
    actual_label = extract_label(example_path)
    print(actual_label, "| expected:", expected_label, "|", example_path)
    assert actual_label == expected_label, (
        f"Label parser mismatch: expected {expected_label}, got {actual_label}"
    )

print("\nAll class-label parser sanity checks passed.")

for example in parser_examples:
    print(
        "Group candidate:",
        extract_group_id_candidate(example[0]),
        "| subtype:",
        extract_subtype(example[0])
    )

NORMAL | expected: NORMAL | srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1003233-0001.jpeg
PNEUMONIA | expected: PNEUMONIA | srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/PNEUMONIA/BACTERIA-266879-0001.jpeg
PNEUMONIA | expected: PNEUMONIA | srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/test/PNEUMONIA/VIRUS-490019-0001.jpeg

All class-label parser sanity checks passed.
Group candidate: ('1003233', 'filename_numeric_group_candidate', 'low') | subtype: None
Group candidate: ('266879', 'filename_numeric_group_candidate', 'low') | subtype: BACTERIAL
Group candidate: ('490019', 'filename_numeric_group_candidate', 'low') | subtype: VIRAL


## 4. Inventory image files

The scan supports an extracted directory or ZIP archive. ZIP members are read directly; the archive is not extracted.

In [4]:
# CELL 4 — SCAN DIRECTORY OR ZIP ARCHIVE
ARCHIVE_HANDLE = None
records = []

if selected_source.is_dir():
    print("Scanning extracted directory...")
    image_paths = (
        p for p in selected_source.rglob("*")
        if p.is_file() and is_image_file(p.name)
    )
    for idx, p in enumerate(tqdm(image_paths, desc="Inventorying images")):
        if MAX_FILES_TO_SCAN is not None and idx >= MAX_FILES_TO_SCAN:
            print("Reached MAX_FILES_TO_SCAN:", MAX_FILES_TO_SCAN)
            break
        rel = p.relative_to(selected_source).as_posix()
        try:
            size = p.stat().st_size
        except OSError:
            size = -1
        records.append(make_record(rel, "directory", size, absolute_path=p))

else:
    print("Scanning ZIP central directory (no extraction)...")
    ARCHIVE_HANDLE = zipfile.ZipFile(selected_source, mode="r")
    image_infos = [
        info for info in ARCHIVE_HANDLE.infolist()
        if not info.is_dir()
        and is_image_file(info.filename)
        and not info.filename.startswith("__MACOSX/")
        and not PurePosixPath(info.filename).name.startswith(".")
    ]
    if MAX_FILES_TO_SCAN is not None:
        image_infos = image_infos[:MAX_FILES_TO_SCAN]

    for info in tqdm(image_infos, desc="Inventorying ZIP members"):
        records.append(
            make_record(info.filename, "zip", info.file_size,
                        archive_member=info.filename)
        )

df_all = pd.DataFrame(records)
if df_all.empty:
    raise RuntimeError(
        "No supported JPEG/PNG/TIFF/BMP images found. Check DATASET_PATH and archive contents."
    )

print("\nALL IMAGE FILES:", f"{len(df_all):,}")
print("Total image data (GB):",
      round(df_all["file_size_bytes"].clip(lower=0).sum() / (1024**3), 3))
print("\nExtensions:")
display(df_all["extension"].value_counts().rename_axis("extension").to_frame("count"))
print("\nModality hints:")
display(df_all["modality_hint"].value_counts().rename_axis("modality").to_frame("count"))
print("\nExample paths:")
display(df_all[["relative_path", "label", "split", "subtype", "modality_hint"]].head(N_EXAMPLE_PATHS))

Scanning extracted directory...


Inventorying images: 0it [00:00, ?it/s]


ALL IMAGE FILES: 115,165
Total image data (GB): 7.881

Extensions:


,count
extension,
.jpeg,115165



Modality hints:


,count
modality,
OCT_OR_OTHER,109309
CHEST_XRAY,5856



Example paths:


,relative_path,label,split,subtype,modality_hint
0,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1003233-0001.jpeg,NORMAL,train,UNKNOWN,CHEST_XRAY
1,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1012843-0001.jpeg,NORMAL,train,UNKNOWN,CHEST_XRAY
2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1014768-0001.jpeg,NORMAL,train,UNKNOWN,CHEST_XRAY
3,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1023731-0001.jpeg,NORMAL,train,UNKNOWN,CHEST_XRAY
4,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1029510-0001.jpeg,NORMAL,train,UNKNOWN,CHEST_XRAY
5,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1031320-0001.jpeg,NORMAL,train,UNKNOWN,CHEST_XRAY
6,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1034239-0001.jpeg,NORMAL,train,UNKNOWN,CHEST_XRAY
7,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1041364-0001.jpeg,NORMAL,train,UNKNOWN,CHEST_XRAY
8,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1044645-0001.jpeg,NORMAL,train,UNKNOWN,CHEST_XRAY
9,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1055663-0001.jpeg,NORMAL,train,UNKNOWN,CHEST_XRAY


## 5. Isolate the chest X-ray subset

The default filter keeps records with a chest X-ray path/filename hint. It excludes obvious OCT/retinal subfolders. Unknown paths are not silently assumed to be chest X-rays.

In [5]:
# CELL 5 — FILTER CHEST X-RAY CANDIDATES
if CHEST_ONLY_FILTER:
    df_cxr = df_all[df_all["chest_xray_candidate"]].copy()
else:
    print("WARNING: filtering is disabled; all image modalities may be included.")
    df_cxr = df_all.copy()

if df_cxr.empty:
    display(df_all[["relative_path", "label", "split", "modality_hint"]].head(50))
    raise RuntimeError(
        "No chest X-ray candidates found. Inspect paths, then set DATASET_PATH "
        "to the exact chest_xray folder or update the modality parser. Do not "
        "disable the filter without confirming the files are chest X-rays."
    )

print("ALL IMAGES:", f"{len(df_all):,}")
print("CHEST X-RAY CANDIDATES:", f"{len(df_cxr):,}")
print("EXCLUDED / UNCERTAIN:", f"{len(df_all)-len(df_cxr):,}")
print("CXR image data (GB):",
      round(df_cxr["file_size_bytes"].clip(lower=0).sum()/(1024**3), 3))

print("\nCandidate labels:")
display(df_cxr["label"].value_counts().rename_axis("label").to_frame("count"))
print("\nCandidate splits:")
display(df_cxr["split"].value_counts().rename_axis("split").to_frame("count"))
print("\nCandidate examples:")
display(df_cxr[["relative_path","label","split","subtype","filename_group_id_candidate",
                "group_id_method","group_id_confidence"]].head(N_EXAMPLE_PATHS))

df_all.to_csv(OUTPUT_DIR / "all_image_inventory.csv", index=False)
df_cxr.to_csv(OUTPUT_DIR / "chest_xray_manifest_preliminary.csv", index=False)

ALL IMAGES: 115,165
CHEST X-RAY CANDIDATES: 5,856
EXCLUDED / UNCERTAIN: 109,309
CXR image data (GB): 1.181

Candidate labels:


,count
label,
PNEUMONIA,4273
NORMAL,1583



Candidate splits:


,count
split,
train,5232
test,624



Candidate examples:


,relative_path,label,split,subtype,filename_group_id_candidate,group_id_method,group_id_confidence
0,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1003233-0001.jpeg,NORMAL,train,UNKNOWN,1003233,filename_numeric_group_candidate,low
1,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1012843-0001.jpeg,NORMAL,train,UNKNOWN,1012843,filename_numeric_group_candidate,low
2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1014768-0001.jpeg,NORMAL,train,UNKNOWN,1014768,filename_numeric_group_candidate,low
3,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1023731-0001.jpeg,NORMAL,train,UNKNOWN,1023731,filename_numeric_group_candidate,low
4,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1029510-0001.jpeg,NORMAL,train,UNKNOWN,1029510,filename_numeric_group_candidate,low
5,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1031320-0001.jpeg,NORMAL,train,UNKNOWN,1031320,filename_numeric_group_candidate,low
6,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1034239-0001.jpeg,NORMAL,train,UNKNOWN,1034239,filename_numeric_group_candidate,low
7,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1041364-0001.jpeg,NORMAL,train,UNKNOWN,1041364,filename_numeric_group_candidate,low
8,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1044645-0001.jpeg,NORMAL,train,UNKNOWN,1044645,filename_numeric_group_candidate,low
9,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-1055663-0001.jpeg,NORMAL,train,UNKNOWN,1055663,filename_numeric_group_candidate,low


## 6. Audit labels, class balance, original splits, and subtype labels

Unknown labels and splits are retained and reported. They are not silently assigned to a class.

In [6]:
# CELL 6 — LABEL, SPLIT AND SUBTYPE DISTRIBUTIONS
class_counts = df_cxr["label"].value_counts(dropna=False).rename_axis("label").reset_index(name="image_count")
class_counts["percentage"] = (class_counts["image_count"] / len(df_cxr) * 100).round(2)

print("PRIMARY CLASS DISTRIBUTION")
display(class_counts)

print("CLASS DISTRIBUTION BY ORIGINAL SPLIT")
split_class_counts = pd.crosstab(df_cxr["split"], df_cxr["label"], margins=True)
display(split_class_counts)

print("BACTERIAL / VIRAL SUBTYPE DISTRIBUTION (PNEUMONIA-LABELLED IMAGES)")
pneumonia_rows = df_cxr[df_cxr["label"] == "PNEUMONIA"]
subtype_counts = pneumonia_rows["subtype"].value_counts(dropna=False).rename_axis("subtype").reset_index(name="image_count")
if len(subtype_counts):
    subtype_counts["percentage_of_pneumonia"] = (
        subtype_counts["image_count"] / max(len(pneumonia_rows), 1) * 100
    ).round(2)
display(subtype_counts)

split_counts = df_cxr["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="image_count")
display(split_counts)

class_counts.to_csv(OUTPUT_DIR / "class_counts.csv", index=False)
split_class_counts.to_csv(OUTPUT_DIR / "class_by_split.csv")
subtype_counts.to_csv(OUTPUT_DIR / "subtype_counts.csv", index=False)
split_counts.to_csv(OUTPUT_DIR / "split_counts.csv", index=False)

print("Unknown labels:", int((df_cxr["label"] == "UNKNOWN").sum()))
print("Unknown splits:", int((df_cxr["split"] == "UNKNOWN").sum()))
print("Bacterial labels:", int((df_cxr["subtype"] == "BACTERIAL").sum()))
print("Viral labels:", int((df_cxr["subtype"] == "VIRAL").sum()))

PRIMARY CLASS DISTRIBUTION


,label,image_count,percentage
0,PNEUMONIA,4273,72.97
1,NORMAL,1583,27.03


CLASS DISTRIBUTION BY ORIGINAL SPLIT


label,NORMAL,PNEUMONIA,All
split,,,
test,234,390,624
train,1349,3883,5232
All,1583,4273,5856


BACTERIAL / VIRAL SUBTYPE DISTRIBUTION (PNEUMONIA-LABELLED IMAGES)


,subtype,image_count,percentage_of_pneumonia
0,BACTERIAL,2780,65.06
1,VIRAL,1493,34.94


,split,image_count
0,train,5232
1,test,624


Unknown labels: 0
Unknown splits: 0
Bacterial labels: 2780
Viral labels: 1493


## 7. Inspect image dimensions, color modes, and readability

Header inspection obtains width, height, mode, and format without resizing images. Set `FULL_DECODE_CHECK=True` in Cell 1 only if you want to fully decode every image; this is slower.

In [7]:
# CELL 7 — IMAGE HEADER / READABILITY AUDIT
@contextmanager
def open_record_stream(row):
    if row["storage_type"] == "zip":
        if ARCHIVE_HANDLE is None:
            raise RuntimeError("ZIP archive handle is not available.")
        stream = ARCHIVE_HANDLE.open(row["archive_member"], "r")
    else:
        stream = open(row["absolute_path"], "rb")
    try:
        yield stream
    finally:
        stream.close()

def inspect_image_record(row):
    result = {
        "width": None, "height": None, "image_mode": None,
        "image_format": None, "aspect_ratio": None,
        "header_read_ok": False, "image_error": None
    }
    try:
        with open_record_stream(row) as stream:
            with Image.open(stream) as img:
                width, height = img.size
                result.update({
                    "width": int(width),
                    "height": int(height),
                    "image_mode": str(img.mode),
                    "image_format": str(img.format),
                    "aspect_ratio": round(width / height, 5) if height else None
                })
                if FULL_DECODE_CHECK:
                    img.load()
                result["header_read_ok"] = True
    except Exception as exc:
        result["image_error"] = f"{type(exc).__name__}: {str(exc)[:250]}"
    return result

if CHECK_IMAGE_HEADERS:
    metadata_rows = []
    for row in tqdm(df_cxr.to_dict("records"), total=len(df_cxr),
                    desc="Inspecting image headers"):
        metadata_rows.append(inspect_image_record(row))

    metadata_df = pd.DataFrame(metadata_rows, index=df_cxr.index)
    df_cxr = pd.concat([df_cxr, metadata_df], axis=1)

    print("Readable image headers:", int(df_cxr["header_read_ok"].sum()))
    print("Failed image headers:", int((~df_cxr["header_read_ok"]).sum()))
    print("\nDimension summary:")
    display(df_cxr.loc[df_cxr["header_read_ok"],
                       ["width","height","aspect_ratio"]].describe().round(3))
    print("\nImage modes:")
    display(df_cxr["image_mode"].value_counts(dropna=False).to_frame("count"))
    print("\nImage formats:")
    display(df_cxr["image_format"].value_counts(dropna=False).to_frame("count"))

    bad = df_cxr[~df_cxr["header_read_ok"]][
        ["relative_path","label","split","image_error"]
    ]
    if len(bad):
        print("Unreadable image examples:")
        display(bad.head(30))
else:
    print("Header inspection disabled.")

Inspecting image headers:   0%|          | 0/5856 [00:00<?, ?it/s]

Readable image headers: 5856
Failed image headers: 0

Dimension summary:


,width,height,aspect_ratio
count,5856.000,5856.000,5856.000
mean,1328.479,974.159,1.440
std,364.019,386.397,0.255
min,384.000,127.000,0.835
25%,1056.000,688.000,1.257
50%,1281.000,888.000,1.412
75%,1560.000,1195.500,1.583
max,2916.000,2713.000,3.379



Image modes:


,count
image_mode,
L,5573
RGB,283



Image formats:


,count
image_format,
JPEG,5856


## 8. Filename-derived group candidates and overlap across splits

This is a potential leakage check, not proof of patient independence. The dataset uses filenames such as `NORMAL-3780378-0001.jpeg`, `BACTERIA-266879-0001.jpeg`, and `VIRUS-490019-0001.jpeg`. The numeric prefix is treated as a filename-derived grouping candidate with low confidence. It may represent a patient, study, or source grouping; it is not confirmed patient metadata.

In [8]:
# CELL 8 — FILENAME-DERIVED GROUP-ID COVERAGE AND SPLIT OVERLAP
group_id_available = df_cxr["filename_group_id_candidate"].notna()
group_id_coverage_percent = round(group_id_available.mean() * 100, 2)

print("Total images:", len(df_cxr))
print("Images with filename-derived group candidate:", int(group_id_available.sum()))
print("Images without group candidate:", int((~group_id_available).sum()))
print("Filename group-candidate coverage (%):", group_id_coverage_percent)

group_df = df_cxr[
    group_id_available & (df_cxr["split"] != "UNKNOWN")
].copy()

if group_df.empty:
    group_split_summary = pd.DataFrame(
        columns=[
            "filename_group_id_candidate", "number_of_splits",
            "splits", "image_count", "group_id_methods", "group_id_confidence"
        ]
    )
    group_split_overlaps = group_split_summary.copy()
    print("No usable filename-derived group candidates plus split labels were recovered.")
else:
    group_split_summary = (
        group_df.groupby("filename_group_id_candidate")
        .agg(
            number_of_splits=("split", "nunique"),
            splits=("split", lambda x: ", ".join(sorted(set(x)))),
            image_count=("relative_path", "count"),
            group_id_methods=("group_id_method", lambda x: ", ".join(sorted(set(x.dropna())))),
            group_id_confidence=("group_id_confidence", lambda x: ", ".join(sorted(set(x.dropna()))))
        )
        .reset_index()
    )

    group_split_overlaps = group_split_summary[
        group_split_summary["number_of_splits"] > 1
    ].copy()

    print("Unique inferred filename groups:", len(group_split_summary))
    print("Filename groups appearing in multiple splits:", len(group_split_overlaps))
    if len(group_split_overlaps):
        display(group_split_overlaps.head(50))
    else:
        print(
            "No cross-split overlap detected among the inferred filename groups. "
            "These are heuristic grouping candidates, not confirmed patient IDs."
        )

group_split_summary.to_csv(
    OUTPUT_DIR / "filename_group_split_summary.csv", index=False
)
group_split_overlaps.to_csv(
    OUTPUT_DIR / "filename_group_split_overlaps.csv", index=False
)

if group_id_coverage_percent < 50:
    print(
        "CAUTION: less than 50% filename-group coverage; "
        "split independence cannot be assessed reliably from filenames."
    )

print(
    "IMPORTANT: Filename-derived group candidates may represent a patient, "
    "study, or source grouping. They do not establish patient identity."
)

Total images: 5856
Images with filename-derived group candidate: 5856
Images without group candidate: 0
Filename group-candidate coverage (%): 100.0
Unique inferred filename groups: 3081
Filename groups appearing in multiple splits: 0
No cross-split overlap detected among the inferred filename groups. These are heuristic grouping candidates, not confirmed patient IDs.
IMPORTANT: Filename-derived group candidates may represent a patient, study, or source grouping. They do not establish patient identity.


## 9. Exact duplicate detection using SHA-256

SHA-256 detects exact byte-for-byte duplicates, including copies stored under different filenames. It does not detect re-encoded or resized visual duplicates.

The notebook hashes only the selected chest X-ray candidates and skips hashing if their count exceeds `MAX_HASH_FILES`.

In [9]:
# CELL 9 — SHA-256 EXACT DUPLICATE CHECK
def sha256_record(row):
    digest = hashlib.sha256()
    with open_record_stream(row) as stream:
        while True:
            chunk = stream.read(HASH_CHUNK_SIZE)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

duplicate_df = pd.DataFrame(
    columns=["sha256","duplicate_count","relative_path","split","label"]
)

if COMPUTE_SHA256 and len(df_cxr) <= MAX_HASH_FILES:
    hashes = []
    for row in tqdm(df_cxr.to_dict("records"), total=len(df_cxr),
                    desc="Computing SHA-256"):
        try:
            hashes.append(sha256_record(row))
        except Exception as exc:
            hashes.append(None)
            print("Hash failed:", row["relative_path"], type(exc).__name__, str(exc)[:120])

    df_cxr["sha256"] = hashes
    valid = df_cxr[df_cxr["sha256"].notna()].copy()
    duplicate_rows = valid.groupby("sha256").filter(lambda group: len(group) > 1)

    if len(duplicate_rows):
        duplicate_df = duplicate_rows[
            ["sha256","relative_path","split","label"]
        ].copy()
        duplicate_df["duplicate_count"] = duplicate_df.groupby("sha256")["sha256"].transform("size")
        duplicate_df = duplicate_df[
            ["sha256","duplicate_count","relative_path","split","label"]
        ]
        print("Duplicate file rows:", len(duplicate_df))
        display(duplicate_df.head(100))

        cross_split = (
            duplicate_rows.groupby("sha256")["split"].nunique().gt(1).sum()
        )
        print("Exact duplicate hash groups crossing splits:", int(cross_split))
    else:
        print("No exact duplicates found among successfully hashed images.")

elif COMPUTE_SHA256:
    df_cxr["sha256"] = None
    print(
        f"Hashing skipped: {len(df_cxr)} candidates exceed MAX_HASH_FILES="
        f"{MAX_HASH_FILES}. Increase the limit only after reviewing the candidate count."
    )
else:
    df_cxr["sha256"] = None
    print("SHA-256 hashing disabled in Cell 1.")

duplicate_df.to_csv(OUTPUT_DIR / "exact_duplicates.csv", index=False)

Computing SHA-256:   0%|          | 0/5856 [00:00<?, ?it/s]

Duplicate file rows: 56


,sha256,duplicate_count,relative_path,split,label
403,3853a4552e0bc9aea1f45b2e7ee27be5954cdf9a0365d7972663b62b5495d9ae,2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-3780378-0001.jpeg,train,NORMAL
404,3853a4552e0bc9aea1f45b2e7ee27be5954cdf9a0365d7972663b62b5495d9ae,2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/NORMAL/NORMAL-3780378-0002.jpeg,train,NORMAL
1801,a0bb88fd5d30acda3316391659ac8d9a8170c668410668f4401b8bc9ce6eff5e,2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/PNEUMONIA/BACTERIA-266879-0001.jpeg,train,PNEUMONIA
1802,a0bb88fd5d30acda3316391659ac8d9a8170c668410668f4401b8bc9ce6eff5e,2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/PNEUMONIA/BACTERIA-266879-0002.jpeg,train,PNEUMONIA
2362,d77e4825fff27bbc2cc15a81dcba699ed8178cd220da72e37872524314fcf8ad,2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/PNEUMONIA/BACTERIA-4615614-0003.jpeg,train,PNEUMONIA
2363,d77e4825fff27bbc2cc15a81dcba699ed8178cd220da72e37872524314fcf8ad,2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/PNEUMONIA/BACTERIA-4615614-0004.jpeg,train,PNEUMONIA
2384,e324d8180008b810b79ccda935c21c2f5c82a12a5bd413fc14ad27df26e062cc,2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/PNEUMONIA/BACTERIA-4647402-0003.jpeg,train,PNEUMONIA
2385,e324d8180008b810b79ccda935c21c2f5c82a12a5bd413fc14ad27df26e062cc,2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/PNEUMONIA/BACTERIA-4647402-0004.jpeg,train,PNEUMONIA
2394,ca083e6fbd862caec218a0686248c3c501b08c21eebb571a629769174edc54c0,2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/PNEUMONIA/BACTERIA-4687423-0002.jpeg,train,PNEUMONIA
2395,ca083e6fbd862caec218a0686248c3c501b08c21eebb571a629769174edc54c0,2,srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray/train/PNEUMONIA/BACTERIA-4687423-0003.jpeg,train,PNEUMONIA


Exact duplicate hash groups crossing splits: 0


## 10. Final audit summary and exported files

The final cell writes a machine-readable JSON summary and the enriched chest X-ray manifest to Kaggle's writable output directory.

In [10]:
# CELL 10 — FINAL AUDIT SUMMARY AND EXPORT
def count_value(series, value):
    return int((series == value).sum())

summary = {
    "project": "PneumoLens",
    "notebook": "Pneumonia",
    "phase": "Phase 2 - Dataset Audit and Split Integrity",
    "selected_source": str(selected_source),
    "source_type": "directory" if selected_source.is_dir() else "zip",
    "all_image_files_found": int(len(df_all)),
    "chest_xray_candidates": int(len(df_cxr)),
    "chest_xray_data_size_gb": round(
        float(df_cxr["file_size_bytes"].clip(lower=0).sum()) / (1024**3), 4
    ),
    "normal_images": count_value(df_cxr["label"], "NORMAL"),
    "pneumonia_images": count_value(df_cxr["label"], "PNEUMONIA"),
    "unknown_label_images": count_value(df_cxr["label"], "UNKNOWN"),
    "train_images": count_value(df_cxr["split"], "train"),
    "validation_images": count_value(df_cxr["split"], "val"),
    "test_images": count_value(df_cxr["split"], "test"),
    "unknown_split_images": count_value(df_cxr["split"], "UNKNOWN"),
    "bacterial_subtype_labels": count_value(df_cxr["subtype"], "BACTERIAL"),
    "viral_subtype_labels": count_value(df_cxr["subtype"], "VIRAL"),
    "filename_group_id_coverage_percent": group_id_coverage_percent,
    "filename_group_overlap_groups": int(len(group_split_overlaps)),
    "image_header_check_performed": bool(CHECK_IMAGE_HEADERS),
    "full_decode_check": bool(FULL_DECODE_CHECK),
    "readable_image_headers": (
        int(df_cxr["header_read_ok"].sum()) if "header_read_ok" in df_cxr else None
    ),
    "unreadable_image_headers": (
        int((~df_cxr["header_read_ok"]).sum()) if "header_read_ok" in df_cxr else None
    ),
    "sha256_enabled": bool(COMPUTE_SHA256),
    "exact_duplicate_rows_reported": int(len(duplicate_df))
}

audit_warnings = []
if summary["unknown_label_images"]:
    audit_warnings.append("Some chest X-ray candidates have UNKNOWN labels.")
if summary["unknown_split_images"]:
    audit_warnings.append("Some images have no recognized train/val/test split.")
if summary["filename_group_id_coverage_percent"] < 50:
    audit_warnings.append("Filename-group coverage is limited; split independence is not established.")
if summary["filename_group_overlap_groups"]:
    audit_warnings.append("Filename-derived group candidates occur in multiple splits; investigate possible leakage.")
if summary["unreadable_image_headers"] not in (None, 0):
    audit_warnings.append("Some images could not be opened by Pillow.")
if summary["bacterial_subtype_labels"] == 0 or summary["viral_subtype_labels"] == 0:
    audit_warnings.append("A complete bacterial-vs-viral subtype task is not supported by recovered labels.")
if summary["validation_images"] < 30:
    audit_warnings.append("Validation split is very small or absent; plan a validation strategy before tuning.")
if summary["test_images"] == 0:
    audit_warnings.append("No test split was identified; define a final test strategy before evaluation.")

summary["warnings"] = audit_warnings

print("=" * 72)
print("PNEUMOLENS — PHASE 2 AUDIT SUMMARY")
print("=" * 72)
for key, value in summary.items():
    print(f"{key}: {value}")

df_cxr.to_csv(OUTPUT_DIR / "chest_xray_manifest.csv", index=False)
with open(OUTPUT_DIR / "audit_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print("\nEXPORTED FILES")
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(f"{p.name} — {p.stat().st_size / 1024:.1f} KB")

print("\nAudit output directory:", OUTPUT_DIR)

PNEUMOLENS — PHASE 2 AUDIT SUMMARY
project: PneumoLens
notebook: Pneumonia
phase: Phase 2 - Dataset Audit and Split Integrity
selected_source: /kaggle/input/datasets
source_type: directory
all_image_files_found: 115165
chest_xray_candidates: 5856
chest_xray_data_size_gb: 1.1814
normal_images: 1583
pneumonia_images: 4273
unknown_label_images: 0
train_images: 5232
validation_images: 0
test_images: 624
unknown_split_images: 0
bacterial_subtype_labels: 2780
viral_subtype_labels: 1493
filename_group_id_coverage_percent: 100.0
filename_group_overlap_groups: 0
image_header_check_performed: True
full_decode_check: False
readable_image_headers: 5856
unreadable_image_headers: 0
sha256_enabled: True
exact_duplicate_rows_reported: 56
warnings: ['Validation split is very small or absent; plan a validation strategy before tuning.']

EXPORTED FILES
all_image_inventory.csv — 34171.5 KB
audit_summary.json — 1.0 KB
chest_xray_manifest.csv — 2544.8 KB
chest_xray_manifest_preliminary.csv — 2000.8 KB
class

## 11. Phase 2 completion checklist

Before moving to Phase 3, review the outputs and confirm:

- [ ] The selected source is the intended dataset.
- [ ] OCT/retinal images have not been mixed into the chest X-ray subset.
- [ ] NORMAL and PNEUMONIA counts match the actual files.
- [ ] Unknown labels and splits have been reviewed.
- [ ] Bacterial/viral labels have been verified before planning a subtype model.
- [ ] Image readability and dimensions have been inspected.
- [ ] Filename-derived group-candidate coverage and cross-split overlap have been reviewed.
- [ ] Exact duplicate results have been reviewed.
- [ ] `chest_xray_manifest.csv` and `audit_summary.json` were saved.

### Important decision before Phase 3

Do not automatically assume that the original train/validation/test folders are safe. If the validation folder is absent, filename-derived group candidates overlap, or reliable grouping IDs cannot be recovered, we will decide on a defensible split strategy before model training.

**Phase 3 — Exploratory Data Analysis (EDA)** will cover class imbalance, representative image grids, dimensions, aspect ratios, pixel distributions, and evidence-based preprocessing decisions.

**Research caution:** This dataset is pediatric and single-source. Results from this dataset alone do not establish external or clinical validity.